In this notebook, we will evaluate the performance of different machine learning models on the wine quality dataset. We will use the following models:
K-Nearest Neighbors, Logistic Regression, Support Vector Machine, Naive Bayes, and Decision Tree. We will evaluate the models using the following metrics:
1. Train Accuracy: The accuracy of the model on the training data.
2. Test Accuracy: The accuracy of the model on the test data.
3. Cross-Validation Accuracy (Mean ± Std): The mean and standard deviation of the accuracy of the model using 5-fold cross-validation.
4. Bias: The difference between the training accuracy and 100%.
5. Variance: The difference between the training accuracy and test accuracy.

In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score




In [18]:
# Load the dataset
df = pd.read_csv('winequality-red-clean.csv')
# Assuming the dataset is loaded into a pandas DataFrame 'df'
X = df.drop('quality', axis=1)  # Features
y = df['quality']  # Target (quality)

Here we will evaluate the performance of the models using all the features in the dataset. We will calculate the train accuracy, test accuracy, cross-validation accuracy, bias, and variance for each model.

In [27]:
# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both train and test sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machine": SVC(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

# Dictionary to store performance results
performance_data = {
    "Model": [],
    "Train Accuracy": [],
    "Test Accuracy": [],
    "Cross-Validation Accuracy (Mean ± Std)": [],
    "Bias": [],
    "Variance": []
}

# Train and evaluate each model
for model_name, model in models.items():
    # Train the model on scaled data
    model.fit(X_train_scaled, y_train)

    # Predict on training data and test data
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    # Accuracy for both train and test sets
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)

    # Cross-validation results (using scaled data)
    cv_scores = cross_val_score(model, scaler.fit_transform(X), y, cv=5)  # 5-fold cross-validation
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    # Store the results in the performance_data dictionary
    performance_data["Model"].append(model_name + " (All Features)")
    performance_data["Train Accuracy"].append(f"{train_accuracy * 100:.2f}%")
    performance_data["Test Accuracy"].append(f"{test_accuracy * 100:.2f}%")
    performance_data["Cross-Validation Accuracy (Mean ± Std)"].append(f"{cv_mean * 100:.2f}% ± {cv_std * 100:.2f}%")

    # Calculate Bias and Variance
    bias = 100 - (train_accuracy * 100)  # Bias is 100 - train accuracy
    variance = (train_accuracy - test_accuracy) * 100  # Variance is the difference between train and test accuracies

    # Append Bias and Variance to the respective lists
    performance_data['Bias'].append(f"{bias:.2f}%")
    performance_data['Variance'].append(f"{variance:.2f}%")

# Convert the performance data dictionary to a DataFrame
performance_df = pd.DataFrame(performance_data)

# Display the results table
performance_df


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,K-Nearest Neighbors (All Features),69.73%,60.62%,52.47% ± 4.68%,30.27%,9.11%
1,Logistic Regression (All Features),60.29%,62.67%,57.55% ± 3.87%,39.71%,-2.38%
2,Support Vector Machine (All Features),68.95%,66.78%,58.30% ± 4.37%,31.05%,2.17%
3,Naive Bayes (All Features),57.63%,54.79%,55.56% ± 3.33%,42.37%,2.84%
4,Decision Tree (All Features),100.00%,58.22%,46.16% ± 2.41%,0.00%,41.78%


 #### K-Nearest Neighbors (All Features) Results:
KNN shows good training performance, but it overfits, as seen by the gap between training and test accuracy. Cross-validation shows significant variability, indicating instability in performance.

#### Logistic Regression (All Features) Results:
Logistic Regression performs similarly on both the training and test sets, showing more generalization. However, it has a high bias (underfitting) and slightly negative variance, indicating stable but potentially insufficient complexity in the model.

#### Support Vector Machine (All Features) Results:
The SVM shows good generalization with similar train and test accuracies. It has low bias (suggesting reasonable complexity) and low variance, meaning it performs consistently across different data subsets.

#### Naive Bayes (All Features) Results:
 Naive Bayes has relatively lower performance in both training and test sets with high bias, indicating underfitting. It has low variance, but overall performance suggests this model might be too simplistic for the dataset.
 
#### Decision Tree (All Features) Results: 
The Decision Tree model overfits significantly, as seen by the perfect training accuracy and large drop in test accuracy. It also has very high variance, indicating it's highly sensitive to specific data points and lacks generalization.

## Model Evaluation with Feature Selection
Feature selection is a critical step in improving model performance by selecting the most relevant features and reducing complexity.Let's see what effect reducing the number of features has on the model performance.

In [20]:
from sklearn.feature_selection import SelectKBest, f_classif
k = 5  # Adjust this number to select more or fewer features
selector = SelectKBest(f_classif, k=k)
X_train_kbest_scaled = selector.fit_transform(X_train_scaled, y_train)
X_test_kbest_scaled = selector.transform(X_test_scaled)

# Train and evaluate each model with the top k features using scaled data
print("\n--- Using Top K Features  ---\n")
for model_name, model in models.items():
    # Train the model on top k scaled features
    model.fit(X_train_kbest_scaled, y_train)

    # Predict on training data and test data
    y_train_pred_kbest = model.predict(X_train_kbest_scaled)
    y_test_pred_kbest = model.predict(X_test_kbest_scaled)

    # Accuracy for both train and test sets
    train_accuracy_kbest = accuracy_score(y_train, y_train_pred_kbest)
    test_accuracy_kbest = accuracy_score(y_test, y_test_pred_kbest)

    # Cross-validation results
    cv_scores = cross_val_score(model, X_train_kbest_scaled, y_train, cv=5)  # 5-fold cross-validation
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    # Calculate Bias and Variance
    bias = 100 - (train_accuracy_kbest * 100)  # Bias is 100 - train accuracy
    variance = (train_accuracy_kbest - test_accuracy_kbest) * 100  # Variance is the difference between train and test accuracies

    # Record the results
    performance_df.loc[len(performance_df.index)] = [
        model_name + ' Top ' + str(k) + ' Features ',
        f"{train_accuracy_kbest * 100:.2f}%",
        f"{test_accuracy_kbest * 100:.2f}%",
        f"{cv_mean * 100:.2f}% ± {cv_std * 100:.2f}%",
        f"{bias:.2f}%",
        f"{variance:.2f}%"
    ]
performance_df


--- Using Top K Features  ---



,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,K-Nearest Neighbors (All Features),69.73%,60.62%,52.47% ± 4.68%,30.27%,9.11%
1,Logistic Regression (All Features),60.29%,62.67%,57.55% ± 3.87%,39.71%,-2.38%
2,Support Vector Machine (All Features),68.95%,66.78%,58.30% ± 4.37%,31.05%,2.17%
3,Naive Bayes (All Features),57.63%,54.79%,55.56% ± 3.33%,42.37%,2.84%
4,Decision Tree (All Features),100.00%,58.22%,46.16% ± 2.41%,0.00%,41.78%
5,K-Nearest Neighbors Top 5 Features,71.10%,59.93%,58.06% ± 2.54%,28.90%,11.17%
6,Logistic Regression Top 5 Features,60.12%,62.67%,58.58% ± 2.89%,39.88%,-2.55%
7,Support Vector Machine Top 5 Features,64.07%,60.96%,60.12% ± 2.56%,35.93%,3.11%
8,Naive Bayes Top 5 Features,59.78%,54.79%,59.69% ± 3.88%,40.22%,4.98%
9,Decision Tree Top 5 Features,100.00%,60.27%,60.21% ± 3.13%,0.00%,39.73%


#### K-Nearest Neighbors Top 5 Features Results:
Small increase in train accuracy but variance increased. Feature reduction led to a more unstable model.

#### Logistic Regression Top 5 Features Results:
Accuracy remained nearly the same. This model is largely unaffected by feature reduction, possibly due to its simplicity.

#### Support Vector Machine Top 5 Features Results:
Accuracy dropped significantly, especially in terms of train accuracy and test accuracy, indicating a large increase in bias and variance.

#### Naive Bayes Top 5 Features Results:
Train accuracy slightly increased, but variance also increased, suggesting it’s more sensitive to the number of features.

#### Decision Tree Top 5 Features Results:
Test accuracy improved with fewer features, showing that feature reduction helped the model generalize better. Variance also decreased, indicating reduced overfitting.

## Finding Optimal Number of Features
Obviously changing the number of features has different impact on different models. Let's find the optimal number of features for each model by evaluating the performance with different numbers of features and selecting the number that gives the best cross-validation score. 

In [21]:

# Feature Selection Function
def find_optimal_features(model, X_scaled, y, max_features=None, scoring='accuracy', cv=5):
    """
    Finds the optimal number of features for a given model using cross-validation.

    Parameters:
    - model: The machine learning model to evaluate (e.g., LogisticRegression, RandomForestClassifier, etc.)
    - X_scaled: The scaled feature matrix (n_samples, n_features)
    - y: The target variable
    - max_features: The maximum number of features to evaluate (default: None, uses all features)
    - scoring: The metric for cross-validation scoring (default: 'accuracy')
    - cv: The number of cross-validation folds (default: 5)

    Returns:
    - optimal_k: The optimal number of features
    - scores: List of cross-validation scores for each number of features
    """
    if max_features is None:
        max_features = X_scaled.shape[1]

    scores = []
    for k in range(1, max_features + 1):
        # Select top k features using SelectKBest
        selector = SelectKBest(f_classif, k=k)
        X_kbest = selector.fit_transform(X_scaled, y)

        # Perform cross-validation with the selected features
        cv_score = cross_val_score(model, X_kbest, y, cv=cv, scoring=scoring).mean()
        scores.append(cv_score)

    # Find the optimal number of features (where cross-validation score is highest)
    optimal_k = np.argmax(scores) + 1  # Adding 1 because indexing starts at 0

    print(f"Optimal number of features: {optimal_k}")
    print(f"Cross-validation scores for each k: {scores}")
    print(f"Max cross-validation score: {max(scores)}")

    return optimal_k, scores






In [22]:
# Using the find_optimal_features function for all models
for model_name, model in models.items():
    print(f"\n--- {model_name} ---")
    
    # Pass the scaled data X_train_scaled instead of X
    optimal_k, _ = find_optimal_features(model, X_train_scaled, y_train, max_features=10)




--- K-Nearest Neighbors ---
Optimal number of features: 7
Cross-validation scores for each k: [0.4836653094163824, 0.5411723707861047, 0.5557169582920657, 0.5806169986427496, 0.5806133303987382, 0.5917721286819999, 0.5960529694435273, 0.5737463776090386, 0.5875059608965187, 0.5660430651846962]
Max cross-validation score: 0.5960529694435273

--- Logistic Regression ---
Optimal number of features: 8
Cross-validation scores for each k: [0.5531968746561021, 0.5651846960859837, 0.5643336634752943, 0.5874729467004145, 0.5857562085029897, 0.5866145776017021, 0.5900517222405635, 0.5943398994901141, 0.5900553904845751, 0.5917757969260116]
Max cross-validation score: 0.5943398994901141

--- Support Vector Machine ---
Optimal number of features: 10
Cross-validation scores for each k: [0.5403396793954733, 0.5703532518983162, 0.5892190308499321, 0.6072337771908588, 0.6012178570118485, 0.6037782913319394, 0.6063644033601114, 0.6132386926378343, 0.6183779024980741, 0.6200909724514875]
Max cross-vali

Even after optimising the number of features, the models are still pretty bad. The best cross-validation score is around 0.58-0.60, which is not very high. This indicates that the models may not be able to capture the underlying patterns in the data effectively. This also doesn't take into account possible overfitting. A big problem is that the dataset is imbalanced, with most wines having a quality of 5 or 6. This makes it harder for the models to learn the patterns for the other classes. We can try to address this issue by balancing the dataset and see if it improves the model performance.

In [28]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to balance the data
smote = SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=4)

for model_name, model in models.items():
    # Apply SMOTE to the training data
    X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train)

    # Train the model on resampled data
    model.fit(X_resampled, y_resampled)

    # Predict on resampled training data and original test data
    y_train_pred = model.predict(X_resampled)
    y_test_pred = model.predict(X_test_scaled)

    # Accuracy scores for both train and test sets
    train_accuracy = accuracy_score(y_resampled, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)

    # Cross-validation on the resampled training set
    cv_scores = cross_val_score(model, X_resampled, y_resampled, cv=5)  # 5-fold cross-validation
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    # Calculate bias and variance
    bias = 100 - (train_accuracy * 100)  # Bias: difference from perfect training accuracy
    variance = (train_accuracy - test_accuracy) * 100  # Variance: difference between training and test accuracy

    # Store the results in the performance dataframe
    performance_df.loc[len(performance_df.index)] = [
        f"{model_name} with SMOTE",
        f"{train_accuracy * 100:.2f}%",
        f"{test_accuracy * 100:.2f}%",
        f"{cv_mean * 100:.2f}% ± {cv_std * 100:.2f}%",
        f"{bias:.2f}%",
        f"{variance:.2f}%"
    ]

# Display the results table
performance_df


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,K-Nearest Neighbors (All Features),69.73%,60.62%,52.47% ± 4.68%,30.27%,9.11%
1,Logistic Regression (All Features),60.29%,62.67%,57.55% ± 3.87%,39.71%,-2.38%
2,Support Vector Machine (All Features),68.95%,66.78%,58.30% ± 4.37%,31.05%,2.17%
3,Naive Bayes (All Features),57.63%,54.79%,55.56% ± 3.33%,42.37%,2.84%
4,Decision Tree (All Features),100.00%,58.22%,46.16% ± 2.41%,0.00%,41.78%
5,K-Nearest Neighbors with SMOTE,86.89%,50.00%,79.74% ± 1.70%,13.11%,36.89%
6,Logistic Regression with SMOTE,66.23%,42.12%,64.94% ± 1.21%,33.77%,24.11%
7,Support Vector Machine with SMOTE,82.53%,58.90%,79.14% ± 1.71%,17.47%,23.63%
8,Naive Bayes with SMOTE,51.76%,35.62%,51.46% ± 1.39%,48.24%,16.15%
9,Decision Tree with SMOTE,100.00%,57.53%,81.07% ± 2.38%,0.00%,42.47%


SMOTE helps in improving model performance for some models (like SVM and KNN), especially in improving Train Accuracy and Cross-Validation Accuracy. However, Test Accuracy tends to drop for most models, indicating overfitting to the synthetic minority class data.

Models like Naive Bayes and Logistic Regression show poor results after applying SMOTE, suggesting that the synthetic data doesn't fit well with these models, which could lead to overfitting or poor generalization.

Decision Tree performed better after applying SMOTE, with a noticeable improvement in cross-validation performance, but the model still suffers from instability (high variance).


These models are not performing well, we can try to improve the model performance by doing hyperparameter tuning. We will do this in following notebooks dedicated to each model. For now lets see some more advanced models like Random Forest and Gradient Boosting.


In [29]:

# Define the models
models = {
    "Random Forest": RandomForestClassifier(random_state=42,class_weight='balanced', ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Dictionary to store performance results
performance_data = {
    "Model": [],
    "Train Accuracy": [],
    "Test Accuracy": [],
    "Cross-Validation Accuracy (Mean ± Std)": [],
    "Bias": [],
    "Variance": []
}

# Train and evaluate each model
for model_name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)

    # Predict on training data and test data
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Accuracy for both train and test sets
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)

    # Cross-validation results (5-fold)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    # Calculate Bias and Variance
    bias = 100 - (train_accuracy * 100)  # Bias as 100 - train accuracy
    variance = (train_accuracy - test_accuracy) * 100  # Variance as the difference between train and test accuracy

    # Store the results in the performance_data dictionary
    performance_df.loc[len(performance_df.index)] = [
        f"{model_name}",
        f"{train_accuracy * 100:.2f}%",
        f"{test_accuracy * 100:.2f}%",
        f"{cv_mean * 100:.2f}% ± {cv_std * 100:.2f}%",
        f"{bias:.2f}%",
        f"{variance:.2f}%"
    ]

# Convert the performance data dictionary to a DataFrame


# Display the results table
performance_df


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,K-Nearest Neighbors (All Features),69.73%,60.62%,52.47% ± 4.68%,30.27%,9.11%
1,Logistic Regression (All Features),60.29%,62.67%,57.55% ± 3.87%,39.71%,-2.38%
2,Support Vector Machine (All Features),68.95%,66.78%,58.30% ± 4.37%,31.05%,2.17%
3,Naive Bayes (All Features),57.63%,54.79%,55.56% ± 3.33%,42.37%,2.84%
4,Decision Tree (All Features),100.00%,58.22%,46.16% ± 2.41%,0.00%,41.78%
5,K-Nearest Neighbors with SMOTE,86.89%,50.00%,79.74% ± 1.70%,13.11%,36.89%
6,Logistic Regression with SMOTE,66.23%,42.12%,64.94% ± 1.21%,33.77%,24.11%
7,Support Vector Machine with SMOTE,82.53%,58.90%,79.14% ± 1.71%,17.47%,23.63%
8,Naive Bayes with SMOTE,51.76%,35.62%,51.46% ± 1.39%,48.24%,16.15%
9,Decision Tree with SMOTE,100.00%,57.53%,81.07% ± 2.38%,0.00%,42.47%


This notebook shows the base performance of some models. The results are rather underwhelming. We have to see how the models perform after hyperparameter tuning and feature selection. We will do this in the following notebooks. A possible reason for the mediocre performance may be the fact we have too many classes.